# 97 - LangChain avanzado: pipeline con memoria y API — Base

Pipeline **simulado** con memoria y consulta a API pública (Wikipedia).

In [ ]:
import time
try:
    import requests; HAS_REQ=True
except Exception:
    HAS_REQ=False

class SimpleMemory:
    def __init__(self): self.store=[]
    def add(self, item): self.store.append({'ts': time.time(), 'item': item})
    def last(self, n=3): return self.store[-n:]

def fetch_wikipedia(term):
    if not HAS_REQ: return {'summary': f'(simulado) Resumen de {term}'}
    try:
        url='https://en.wikipedia.org/api/rest_v1/page/summary/'+term
        r=requests.get(url,timeout=10)
        if r.status_code==200:
            data=r.json(); return {'summary': data.get('extract','(sin extract)')}
        return {'summary': f'(fallback) No encontrado: {term}'}
    except Exception:
        return {'summary': f'(offline) No se pudo consultar {term}'}

def pipeline(question, memory: SimpleMemory):
    q=question.strip(); memory.add({'user_question': q})
    term=q.split()[-1].capitalize()
    info=fetch_wikipedia(term)
    answer=f'Pregunta: {q}\nEntidad: {term}\nInfo: {info["summary"][:200]}...'
    memory.add({'answer': answer})
    return answer

mem=SimpleMemory(); print(pipeline('¿Qué es Python?', mem)); print('Memoria:', mem.last())


## Ampliación progresiva

Un pipeline avanzado suele necesitar memoria, caché, extracción de entidad y fallback offline. El siguiente ejemplo mantiene la idea de LangChain sin depender de una API de pago.

Esta celda añade funciones auxiliares para extraer la entidad principal de la pregunta y consultar Wikipedia con caché. Son pasos previos al pipeline avanzado.


In [ ]:
def extract_entity(question):
    q = question.strip().strip('¿?')
    tokens = [token.strip('.,:;()') for token in q.split()]
    candidates = [token for token in tokens if token and token[0].isupper()]
    return candidates[-1] if candidates else tokens[-1].capitalize()

class CachedWikipedia:
    def __init__(self):
        self.cache = {}
    def get(self, term):
        if term not in self.cache:
            self.cache[term] = fetch_wikipedia(term)
        return self.cache[term]

wiki = CachedWikipedia()
print(extract_entity('¿Qué es Python?'))
print(wiki.get('Python')['summary'][:120])

Esta celda une los pasos avanzados del pipeline: limpieza de pregunta, extracción de entidad, consulta con caché, actualización de memoria y generación de una respuesta trazable.


In [ ]:
def advanced_pipeline(question, memory):
    q = question.strip()
    entity = extract_entity(q)
    info = wiki.get(entity)
    previous = memory.last(2)
    answer = (
        f"Pregunta: {q}\n"
        f"Entidad detectada: {entity}\n"
        f"Contexto externo: {info['summary'][:220]}...\n"
        f"Memoria reciente: {previous}\n"
        "Respuesta simulada: combina contexto externo y conversación previa."
    )
    memory.add({'question': q, 'entity': entity, 'answer': answer})
    return answer

mem2 = SimpleMemory()
print(advanced_pipeline('¿Qué es Python?', mem2))
print(advanced_pipeline('¿Y para qué se usa Python?', mem2))

### Reto adicional

Añade un paso que detecte el idioma de la pregunta y cambie el idioma de la respuesta simulada.

## Para profundizar
Este notebook introduce LangChain avanzado. La documentación completa incluye:
- Tools, Agents, Memory, RAG, Output Parsers

Consulta el documento correspondiente en Documentacion/.

In [ ]:
# Los ejemplos avanzados están en los documentos de Documentacion/# Consulta el archivo correspondiente según lo que quieras profundizar.